# Ajuste reproducible del tutor
Este notebook no contiene resultados de entrenamiento. Necesita un dataset aprobado, una selección documentada de modelo y acceso al hardware adecuado. No aprueba datos ni modelos automáticamente. El conjunto test se reserva para evaluación final. No escriba tokens, contraseñas ni datos privados en las celdas.


In [ ]:
import os, re, subprocess, sys, secrets
from pathlib import Path
revision = os.environ['BACKEND_COMMIT']
assert re.fullmatch(r'[0-9a-f]{40}', revision), 'Fije el commit de código entregado.'
root = Path(os.environ.get('BACKEND_DIRECTORY', '/content/lizardjudge-backend'))
if not root.exists():
    subprocess.run(['git', 'clone', 'https://github.com/BenjaminSM04/PS2B2026-JuezVirtual_Backend', str(root)], check=True)
assert not subprocess.check_output(['git', '-C', str(root), 'status', '--porcelain']).strip(), 'Conserve los cambios existentes en otro checkout.'
subprocess.run(['git', '-C', str(root), 'fetch', 'origin', revision], check=True)
subprocess.run(['git', '-C', str(root), 'checkout', '--detach', revision], check=True)
os.chdir(root)


In [ ]:
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'libpq-dev', 'build-essential'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'deploy/requirements.txt', '-r', 'deploy/requirements-training.txt'], check=True)
config = root / 'data/config'
config.mkdir(parents=True, exist_ok=True)
key = config / 'secret.key'
if not key.exists():
    key.write_text(secrets.token_urlsafe(64))
os.environ['OJ_ENV'] = 'dev'
os.environ['POSTGRES_HOST'] = '127.0.0.1'
os.environ['POSTGRES_PORT'] = '1'


## Entradas externas
Defina DATASET_DIR, MODEL_SELECTION y TRAIN_OUTPUT mediante el entorno de Colab/Kaggle. DATASET_DIR debe tener manifest.json y train/validation/test.jsonl generados por tutor_dataset con aprobación humana válida. MODEL_SELECTION debe registrar modelo, licencia, revisión inmutable, benchmark y revisor. HF_TOKEN, si hace falta, se proporciona exclusivamente mediante el gestor de secretos del entorno. No se ejecutan migraciones ni se accede a una base del servidor.


In [ ]:
os.environ['DJANGO_SETTINGS_MODULE'] = 'oj.settings'
import django
django.setup()
from ai_tutor.dataset.pipeline import load_dataset
dataset = Path(os.environ['DATASET_DIR'])
parts, manifest = load_dataset(dataset)
assert all(parts[name] for name in ('train', 'validation', 'test'))
selection = Path(os.environ['MODEL_SELECTION'])
output = Path(os.environ['TRAIN_OUTPUT'])
import torch
hardware = {'cuda': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}
print(hardware)


In [ ]:
from ai_tutor.ml.training import train_model
result = train_model(dataset, selection, output, resume=os.environ.get('RESUME') == '1', quantized=os.environ.get('USE_4BIT', '1') == '1')
print({'dataset_version': result['dataset_version'], 'training_completed': True})


## Artefactos y revisión
Conserve adapter/, checkpoints, training-input.json, training-result.json y environment.txt fuera de Git. Los resultados provienen únicamente de la ejecución real de Trainer. Después se requiere evaluar test por separado, contrastar utilidad/corrección/progresividad/fuga con docentes y decidir integrar, corregir o rechazar. El adaptador no se sirve automáticamente: prepare la conversión GGUF compatible y el manifiesto con checksums y revisión PB-16 antes de habilitar el motor.
